# Day 3 · 5교시 [실습 보조] Claude Agent SDK 최소 예제 — `05_agent_sdk`

## 실습 목표

CLI(`claude`)·헤드리스(`claude -p`, 4교시)를 넘어, **에이전트를 파이썬 코드에 임베드**하는
**Claude Agent SDK**의 최소 형태를 본다. "Claude Code를 파이프라인 부품으로"(3일차 목표)의 종착점.

| 순서 | 내용 | 교안 |
|------|------|------|
| 1 | 설치·키 확인(그레이스풀) | 5.1·5.2 |
| 2 | `query()` — 한 번 물어보기 | 5.2·5.3 |
| 3 | `ClaudeAgentOptions` — 시스템 프롬프트·도구·권한을 코드로 | 5.3 |
| 4 | 커스텀 도구(`@tool`)·MCP·서브에이전트 | 5.4 |
| 5 | 관통 프로젝트: 요약 단계를 SDK로 임베드 | 5.6 |

> ⚠️ **이 영역은 변동이 잦다**(패키지명이 *Claude Code SDK → Claude Agent SDK* 로 바뀐 이력).
> 실습 전 **공식 문서로 패키지명·옵션을 최신화**할 것.
> 준비: `pip install claude-agent-sdk` (**Python 3.10+**) + `ANTHROPIC_API_KEY`.
> **이 노트북은 SDK/키가 없어도 무에러로 실행**된다(코드는 정의만 되고, 실제 호출은 준비됐을 때만).
> MLAPI(OpenAI 호환 게이트웨이)는 이 SDK와 프로토콜이 달라 그대로는 안 되고, Anthropic 키가 필요하다.

In [1]:
import os, inspect

try:
    from claude_agent_sdk import query, ClaudeAgentOptions   # 핵심 진입점
    HAVE_SDK = True
except Exception:
    HAVE_SDK = False

HAVE_KEY = bool(os.getenv("ANTHROPIC_API_KEY"))
READY = HAVE_SDK and HAVE_KEY

print("claude-agent-sdk 설치:", HAVE_SDK)
print("ANTHROPIC_API_KEY   :", HAVE_KEY)
print("→ 실제 실행 가능:", READY, "" if READY else "(아래는 '참고 코드' — 설치+키 시 그대로 실행)")
if not HAVE_SDK:
    print("   설치: pip install claude-agent-sdk   (Python 3.10+)")

claude-agent-sdk 설치: False
ANTHROPIC_API_KEY   : False
→ 실제 실행 가능: False (아래는 '참고 코드' — 설치+키 시 그대로 실행)
   설치: pip install claude-agent-sdk   (Python 3.10+)


## 2. `query()` — 한 번 물어보기 (원샷)

가장 단순한 형태. `query(prompt=...)`는 **비동기 이터레이터**로 메시지를 스트리밍한다.
CLI의 `claude -p "..."`(4교시)를 **코드 안에서** 부르는 것과 같다.

In [2]:
async def ask_once(prompt: str) -> str:
    """SDK 최소 예제 — 한 번 물어보고 텍스트를 모아 반환."""
    out = []
    async for message in query(prompt=prompt):        # 메시지 스트림
        # message 안의 텍스트 블록을 모은다(버전에 따라 접근법이 다를 수 있음 → 공식 문서)
        for block in getattr(message, "content", []) or []:
            text = getattr(block, "text", None)
            if text:
                out.append(text)
    return "".join(out)

# 정의는 SDK 없이도 된다(호출 시점에만 query 필요). 실제 실행은 준비됐을 때만:
if READY:
    import asyncio
    print(asyncio.run(ask_once("한 문장으로: RAG가 뭐야?")))
else:
    print("[참고 코드] ask_once — 설치+키 시 실행됨:\n")
    print(inspect.getsource(ask_once))

[참고 코드] ask_once — 설치+키 시 실행됨:

async def ask_once(prompt: str) -> str:
    """SDK 최소 예제 — 한 번 물어보고 텍스트를 모아 반환."""
    out = []
    async for message in query(prompt=prompt):        # 메시지 스트림
        # message 안의 텍스트 블록을 모은다(버전에 따라 접근법이 다를 수 있음 → 공식 문서)
        for block in getattr(message, "content", []) or []:
            text = getattr(block, "text", None)
            if text:
                out.append(text)
    return "".join(out)



## 3. `ClaudeAgentOptions` — 시스템 프롬프트·도구·권한을 코드로

CLI에서 `CLAUDE.md`·`settings.json`으로 하던 설정(시스템 프롬프트·허용 도구·권한 모드)을
**코드로** 지정한다 — Day1의 harness를 코드 레벨로 내린 셈.

In [3]:
def build_options():
    """Day1 설정 파일·권한(5교시)을 SDK 옵션으로 — 코드가 곧 harness."""
    return ClaudeAgentOptions(
        system_prompt="너는 한국어 요약기다. 사실만, 두 문장 이내.",
        allowed_tools=["Read", "Grep"],        # 최소 권한(Day1 5교시 원칙을 코드로)
        permission_mode="acceptEdits",          # 또는 'default'/'plan' 등
        model="claude-sonnet-5",               # 모델 티어링(Day2 1교시)도 코드로
        # mcp_servers=..., agents=...           # 4절 참고
    )

print("[참고] ClaudeAgentOptions 로 시스템 프롬프트·allowed_tools·permission_mode·model 을 코드 지정")
if HAVE_SDK:
    print("옵션 객체 생성 OK:", type(build_options()).__name__)
else:
    print(inspect.getsource(build_options))

[참고] ClaudeAgentOptions 로 시스템 프롬프트·allowed_tools·permission_mode·model 을 코드 지정
def build_options():
    """Day1 설정 파일·권한(5교시)을 SDK 옵션으로 — 코드가 곧 harness."""
    return ClaudeAgentOptions(
        system_prompt="너는 한국어 요약기다. 사실만, 두 문장 이내.",
        allowed_tools=["Read", "Grep"],        # 최소 권한(Day1 5교시 원칙을 코드로)
        permission_mode="acceptEdits",          # 또는 'default'/'plan' 등
        model="claude-sonnet-5",               # 모델 티어링(Day2 1교시)도 코드로
        # mcp_servers=..., agents=...           # 4절 참고
    )



## 4. 커스텀 도구·MCP·서브에이전트 — 코드로 확장

Day2에서 MCP 서버를 `claude mcp add` 로 붙였다면, SDK에선 **인프로세스 도구**를 `@tool`로 만들어
`create_sdk_mcp_server` 로 묶고, `agents=` 로 서브에이전트(Day2 7교시)를 코드로 정의한다.
(아래는 형태만 — 시그니처는 공식 문서로 확인)

In [4]:
# 형태 예시 (참고): 인프로세스 커스텀 도구 → SDK MCP 서버 → 옵션에 연결
SNIPPET = '''
from claude_agent_sdk import tool, create_sdk_mcp_server, ClaudeAgentOptions

@tool("word_count", "텍스트 단어 수", {"text": str})
async def word_count(args):
    n = len(args["text"].split())
    return {"content": [{"type": "text", "text": f"{n} words"}]}

server = create_sdk_mcp_server(name="tools", tools=[word_count])
options = ClaudeAgentOptions(
    mcp_servers={"tools": server},
    allowed_tools=["mcp__tools__word_count"],
    agents={  # 서브에이전트(Day2 7교시)를 코드로
        "reviewer": {"description": "코드 리뷰 전용", "prompt": "너는 리뷰어다", "tools": ["Read", "Grep"]},
    },
)
'''
print("[참고 코드] @tool → create_sdk_mcp_server → ClaudeAgentOptions(mcp_servers=, agents=)")
print(SNIPPET)

[참고 코드] @tool → create_sdk_mcp_server → ClaudeAgentOptions(mcp_servers=, agents=)

from claude_agent_sdk import tool, create_sdk_mcp_server, ClaudeAgentOptions

@tool("word_count", "텍스트 단어 수", {"text": str})
async def word_count(args):
    n = len(args["text"].split())
    return {"content": [{"type": "text", "text": f"{n} words"}]}

server = create_sdk_mcp_server(name="tools", tools=[word_count])
options = ClaudeAgentOptions(
    mcp_servers={"tools": server},
    allowed_tools=["mcp__tools__word_count"],
    agents={  # 서브에이전트(Day2 7교시)를 코드로
        "reviewer": {"description": "코드 리뷰 전용", "prompt": "너는 리뷰어다", "tools": ["Read", "Grep"]},
    },
)



## 5. 관통 프로젝트 — 요약(Summarizer)을 SDK로 임베드

관통 파이프라인(수집→저장→요약→알림)의 **요약 단계**를, CLI 헤드리스가 아니라 **SDK 함수**로.
그러면 파이프라인 전체가 하나의 파이썬 프로그램 안에서 돈다 — `claude -p`(4교시)가 *셸 명령*이라면
SDK는 *코드 안의 함수*다.

In [5]:
async def summarize(items_text: str) -> str:
    """관통 프로젝트 Summarizer — SDK 임베드 버전(요약 에이전트를 함수로)."""
    opts = ClaudeAgentOptions(system_prompt="수집 항목들을 한국어 3문장으로 요약하라. 사실만.")
    out = []
    async for message in query(prompt=items_text, options=opts):
        for block in getattr(message, "content", []) or []:
            if getattr(block, "text", None):
                out.append(block.text)
    return "".join(out)

SAMPLE = "- MCP 표준이 확산 중\n- 에이전트 병렬 워크플로가 늘어남\n- 보안(프롬프트 인젝션) 관심 증가"
if READY:
    import asyncio
    print(asyncio.run(summarize(SAMPLE)))
else:
    print("[참고 코드] summarize — 파이프라인의 요약 단계를 SDK로:\n")
    print(inspect.getsource(summarize))

[참고 코드] summarize — 파이프라인의 요약 단계를 SDK로:

async def summarize(items_text: str) -> str:
    """관통 프로젝트 Summarizer — SDK 임베드 버전(요약 에이전트를 함수로)."""
    opts = ClaudeAgentOptions(system_prompt="수집 항목들을 한국어 3문장으로 요약하라. 사실만.")
    out = []
    async for message in query(prompt=items_text, options=opts):
        for block in getattr(message, "content", []) or []:
            if getattr(block, "text", None):
                out.append(block.text)
    return "".join(out)



## 실습 정리

- **`query()` + `ClaudeAgentOptions`** = SDK 최소 형태 — 시스템 프롬프트·허용 도구·권한·모델을 **코드로**.
- CLI/헤드리스가 "에이전트를 *셸 명령*으로" 썼다면, SDK는 "에이전트를 *코드 안의 함수*로" 만든다(4교시 종착점).
- MCP·서브에이전트(Day2)도 SDK에서 코드로 — harness를 코드 레벨로 내린 것.
- **CLI vs SDK**: 붙어서 개발·탐색 = CLI / 제품·서비스에 에이전트를 임베드 = SDK.
- ⚠️ 패키지·API 변동이 잦다 → **공식 문서 최신화 필수**. 이 노트북은 형태·감을 잡는 참고용(설치+`ANTHROPIC_API_KEY` 시 그대로 실행).